# Pemodelan Deret Waktu Kasus Malaria Konfirmasi di Provinsi Sumatera Selatan Tahun 2021-2026

**Metode: Single Exponential Smoothing (SES)**

Notebook ini mengolah data kasus malaria konfirmasi mingguan dari 17 kabupaten/kota di Sumatera Selatan (2021 sampai minggu 39 tahun 2026), lalu meramal 8 minggu ke depan.

Alurnya:
1. Statistik deskriptif dan plot deret waktu
2. Split data: 80% training, 20% testing (kronologis, bukan acak)
3. Model SES dilatih di data training, alpha dioptimasi otomatis
4. Model meramal periode testing, hasilnya dievaluasi dengan MAE, RMSE, MSE
5. Model dibangun ulang pakai seluruh data, lalu forecast 8 minggu ke depan

Cara pakai di Google Colab: jalankan cell satu per satu dari atas ke bawah (Shift+Enter).


## 1. Persiapan library

In [ ]:
# install sekali saja (di Colab)
# !pip install -q statsmodels

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing
print('library siap')

## 2. Load data

File yang dibutuhkan: `malaria_sumsel_tidy.csv` (kolom: kab, tahun, pekan, kasus).
Data diambil otomatis dari GitHub (tidak perlu upload manual). Kalau dijalankan lokal tanpa internet, file dibaca dari folder data/.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/dandienter/skripsi-malaria-sumsel/master/data/malaria_sumsel_tidy.csv'
try:
    df = pd.read_csv(DATA_URL)
    print('data diambil otomatis dari GitHub, tidak perlu upload manual')
except Exception:
    try:
        from google.colab import files
        print('upload file malaria_sumsel_tidy.csv')
        up = files.upload()
        df = pd.read_csv(list(up.keys())[0])
    except ImportError:
        df = pd.read_csv('data/malaria_sumsel_tidy.csv')  # mode lokal

print('baris:', len(df))
print('kab/kota:', df['kab'].nunique())
print('rentang:', df['tahun'].min(), 's.d.', df['tahun'].max())
df.head()

## 3. Statistik deskriptif per kabupaten/kota

Tabel ini menjawab pertanyaan paling dasar: daerah mana yang kasusnya paling banyak, dan seberapa "kosong" datanya (persen minggu dengan nol kasus).

In [ ]:
desc = df.groupby('kab')['kasus'].agg(
    total='sum', rata2='mean', maksimum='max',
    minggu_nol=lambda x: round((x == 0).mean() * 100, 1)
).sort_values('total', ascending=False)
desc.columns = ['Total kasus', 'Rata-rata/minggu', 'Maks/minggu', '% minggu nol']
desc

**Cara baca:** kalau `% minggu nol` di atas 90%, artinya datanya sparse (dominan nol). Ini normal untuk data malaria di daerah yang sudah mendekati eliminasi, bukan kesalahan input.

## 4. Plot deret waktu 17 kabupaten/kota

Plot ini untuk memastikan data tidak punya tren dan tidak punya pola musiman, karena itu syarat pemakaian SES.

In [ ]:
kabs = sorted(df['kab'].unique())
fig, axes = plt.subplots(9, 2, figsize=(14, 22))
axes = axes.flatten()
for ax, kab in zip(axes, kabs):
    s = df[df['kab'] == kab].sort_values(['tahun', 'pekan'])
    ax.plot(np.arange(len(s)), s['kasus'].values, lw=1)
    ax.set_title(kab, fontsize=10)
    ax.set_xlabel('Minggu ke-')
    ax.set_ylabel('Kasus')
    ax.grid(alpha=0.3)
axes[-1].axis('off')  # 17 kab, 1 slot kosong
fig.suptitle('Deret waktu kasus malaria konfirmasi per kab/kota (2021-2026)', fontsize=13)
fig.tight_layout()
plt.show()

## 5. Split training-testing (kronologis)

Data deret waktu tidak boleh diacak. Urutannya dipertahankan: **80% minggu pertama** jadi data training, **20% minggu terakhir** jadi data testing.

* Data training: untuk melatih model dan mencari alpha terbaik.
* Data testing: untuk menguji. Model yang sudah jadi disuruh meramal periode testing, lalu ramalannya dibandingkan dengan data aslinya. Selisihnya adalah error yang jujur (out-of-sample), bukan error yang dihitung dari data yang sama dengan data latih.

In [ ]:
def split_train_test(y, train_ratio=0.8):
    n = len(y)
    n_train = int(n * train_ratio)
    return y[:n_train], y[n_train:], n_train

contoh = df[df['kab'] == 'Kota Palembang'].sort_values(['tahun', 'pekan'])['kasus'].values
tr, ts, n_tr = split_train_test(contoh)
print(f'total {len(contoh)} minggu -> training {len(tr)} minggu, testing {len(ts)} minggu')

## 6. Model Single Exponential Smoothing (SES)

SES meramal dengan rumus: ramalan minggu depan = alpha x data minggu ini + (1 - alpha) x ramalan minggu ini.

* **alpha** (0 sampai 1) mengatur seberapa cepat model mengikuti data terbaru. Alpha kecil artinya model "cuek", lebih percaya rata-rata jangka panjang.
* Di sini alpha **dioptimasi otomatis** (bukan diisi manual), jadi nilainya dipilih yang paling cocok untuk tiap kab/kota.
* Karena SES tidak punya komponen tren, hasil ramalannya **datar (konstan)**. Itu sifat metodenya, bukan error.

In [ ]:
def fit_ses(train):
    """Latih SES di data training. Return dict berisi alpha dan fitted values."""
    train = np.asarray(train, dtype=float)
    if train.max() == 0:
        # deret yang isinya nol semua: level = 0, tidak perlu optimasi
        return {'alpha': 0.0, 'fitted': np.zeros_like(train), 'model': None}
    m = SimpleExpSmoothing(train, initialization_method='estimated').fit(optimized=True)
    return {'alpha': float(m.params['smoothing_level']),
            'fitted': np.asarray(m.fittedvalues), 'model': m}

def mae(a, b):  return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
def mse(a, b):  return float(np.mean((np.asarray(a) - np.asarray(b)) ** 2))

print('fungsi siap')

## 7. Evaluasi di data testing

Untuk tiap kab/kota: latih SES di data training, ramal sepanjang periode testing, hitung MAE, RMSE, MSE dari selisih ramalan vs data asli.

**Kenapa tidak pakai MAPE?** MAPE membagi error dengan nilai aktual. Karena data ini 96% nol, pembagian dengan nol menghasilkan tak hingga (Inf). Jadi MAPE tidak valid untuk data ini dan tidak dipakai.

In [ ]:
hasil = []
for kab in kabs:
    y = df[df['kab'] == kab].sort_values(['tahun', 'pekan'])['kasus'].values.astype(float)
    train, test, n_train = split_train_test(y)
    r = fit_ses(train)
    fc_test = np.zeros(len(test)) if r['model'] is None else np.asarray(r['model'].forecast(len(test)))
    hasil.append({
        'Kab/Kota': kab,
        'Alpha (training)': round(r['alpha'], 6),
        'MAE': round(mae(test, fc_test), 4),
        'RMSE': round(rmse(test, fc_test), 4),
        'MSE': round(mse(test, fc_test), 4),
    })

ev = pd.DataFrame(hasil).sort_values('RMSE')
ev

### 7b. Dua alpha: alpha training vs alpha full

Di notebook ini ada **dua alpha**, dan itu memang disengaja:

1. **Alpha training**: hasil optimasi memakai data training saja (239 minggu pertama). Alpha ini dipakai untuk meramal periode testing, supaya evaluasinya jujur (model tidak pernah melihat data testing).
2. **Alpha full**: hasil optimasi ulang memakai **seluruh data** (299 minggu). Alpha ini dipakai untuk forecast final 8 minggu ke depan, karena untuk meramal masa depan kita ingin memakai semua informasi yang ada.

Setiap kali data pelatihannya berubah, alpha dioptimasi ulang. Itu sebabnya nilainya bisa sedikit berbeda.

In [ ]:
alpha_rows = []
for kab in kabs:
    y = df[df['kab'] == kab].sort_values(['tahun', 'pekan'])['kasus'].values.astype(float)
    train, test, n_train = split_train_test(y)
    r_train = fit_ses(train)
    r_full = fit_ses(y)
    alpha_rows.append({'Kab/Kota': kab,
                       'Alpha training': round(r_train['alpha'], 6),
                       'Alpha full': round(r_full['alpha'], 6)})

pd.DataFrame(alpha_rows)

### 7c. Kelayakan model: berapa cutoff MAE/RMSE/MSE yang "layak"?

Jawaban jujurnya: **tidak ada cutoff universal**. MAE 0,5 bisa bagus untuk satu dataset dan jelek untuk dataset lain. Yang benar:

1. **Bandingkan dengan skala data.** Kalau rata-rata kasus cuma 0,3 per minggu, MAE 0,4 artinya error-nya lebih besar dari nilai yang diramal. Kalau rata-rata kasus 50 per minggu, MAE 0,4 itu kecil.
2. **Bandingkan dengan model baseline sederhana.** Model SES baru bisa dibilang berguna kalau error-nya lebih kecil dari cara paling naif sekalipun.
3. **Waspadai data nol.** Data ini 96% nol, jadi error kecil bisa menipu: model yang selalu meramal nol pun error-nya kecil.

Cell di bawah menghitung baseline **naive** (ramalan = nilai minggu terakhir training, datar untuk semua horizon testing) lalu membandingkannya dengan SES.

In [ ]:
base_rows = []
for kab in kabs:
    y = df[df['kab'] == kab].sort_values(['tahun', 'pekan'])['kasus'].values.astype(float)
    train, test, n_train = split_train_test(y)
    r = fit_ses(train)
    fc_ses = np.zeros(len(test)) if r['model'] is None else np.asarray(r['model'].forecast(len(test)))
    fc_naive = np.full(len(test), train[-1])  # baseline: nilai terakhir training
    base_rows.append({'Kab/Kota': kab,
                       'MAE SES': round(mae(test, fc_ses), 4),
                       'MAE naive': round(mae(test, fc_naive), 4),
                       'RMSE SES': round(rmse(test, fc_ses), 4),
                       'RMSE naive': round(rmse(test, fc_naive), 4)})

base = pd.DataFrame(base_rows)
print('Rata-rata MAE  SES  :', round(base['MAE SES'].mean(), 4))
print('Rata-rata MAE  naive:', round(base['MAE naive'].mean(), 4))
print('Rata-rata RMSE SES  :', round(base['RMSE SES'].mean(), 4))
print('Rata-rata RMSE naive:', round(base['RMSE naive'].mean(), 4))
print('SES menang di', int((base['MAE SES'] < base['MAE naive']).sum()), 'dari', len(base), 'kab/kota (MAE)')
base

**Cara baca hasil di atas:** kalau rata-rata MAE SES lebih kecil dari MAE naive, model SES layak dipakai karena mengalahkan cara paling sederhana sekalipun.

**Hasil sebenarnya di data ini (jujur, bukan yang diharapkan):** rata-rata MAE SES = 0,2526 vs naive = 0,1245; rata-rata RMSE SES = 0,3682 vs naive = 0,2951. Naive menang MAE di 16 dari 17 kab/kota (hanya Kota Palembang yang dimenangkan SES, karena di sana nilai terakhir training = 1 sehingga naive meramal 1,0 terus-menerus).

Ini bukan kesalahan hitung. Data testing didominasi minggu nol, dan nilai terakhir training di hampir semua kab/kota juga nol, sehingga strategi naive "selalu ramal nol" hampir tak terkalahkan pada metrik error. Untuk data se-sparse ini, **tidak ada** model yang bisa mengalahkan "ramal nol" pada MAE/RMSE.

Tapi naive itu rapuh: ramalannya 100% bergantung pada satu observasi terakhir. Kalau minggu terakhir training kebetulan spike, naive akan meramal spike itu selama 60 minggu berturut-turut. SES memberi level kecil yang principled dan robust, lebih masuk akal untuk perencanaan (menyiapkan kemungkinan adanya kasus, bukan nol mutlak).

**Kesimpulan kelayakan:** model layak dipakai sebagai peramal level yang stabil dan robust untuk perencanaan, bukan karena mengalahkan naive pada MAE. Jangan klaim "model layak" hanya karena MAE-nya kecil.

## 8. Forecast 8 minggu ke depan

Setelah model terbukti oke di data testing, model **dibangun ulang memakai seluruh data** (training + testing, 2021 sampai minggu 39 tahun 2026), lalu meramal 8 minggu berikutnya: minggu 40 sampai 47 tahun 2026.

In [ ]:
fc8_rows = []
for kab in kabs:
    y = df[df['kab'] == kab].sort_values(['tahun', 'pekan'])['kasus'].values.astype(float)
    r = fit_ses(y)
    fc = np.zeros(8) if r['model'] is None else np.asarray(r['model'].forecast(8))
    for i, v in enumerate(fc, start=1):
        fc8_rows.append({'Kab/Kota': kab, 'Minggu ke': i,
                         'Pekan 2026': 39 + i, 'Forecast': round(float(v), 4)})

fc8 = pd.DataFrame(fc8_rows)
fc8_pivot = fc8.pivot(index='Kab/Kota', columns='Minggu ke', values='Forecast')
fc8_pivot.columns = [f'M+{c} (pekan {39 + c})' for c in fc8_pivot.columns]
fc8_pivot

## 9. Plot lengkap per kabupaten/kota

Biru = data aktual. Merah = fitted value (hasil latih di training). Hijau putus-putus = ramalan periode testing. Oranye = forecast 8 minggu ke depan. Garis titik-titik = batas training/testing.

In [ ]:
for kab in kabs:
    s = df[df['kab'] == kab].sort_values(['tahun', 'pekan']).reset_index(drop=True)
    y = s['kasus'].values.astype(float)
    n = len(y)
    train, test, n_train = split_train_test(y)
    t = np.arange(n)

    fig, ax = plt.subplots(figsize=(12, 4.2))
    ax.plot(t, y, color='#1f77b4', lw=1.2, label='Data aktual')
    r_tr = fit_ses(train)
    if r_tr['model'] is None:
        fc_test = np.zeros(len(test)); fc8v = np.zeros(8)
        ax.axhline(0, color='red', lw=1.2, label='Fitted (level = 0)')
    else:
        ax.plot(t[:n_train], r_tr['fitted'], color='red', lw=1.2, label='Fitted value (training)')
        fc_test = np.asarray(r_tr['model'].forecast(len(test)))
        r_full = fit_ses(y)
        fc8v = np.zeros(8) if r_full['model'] is None else np.asarray(r_full['model'].forecast(8))
    ax.plot(t[n_train:], fc_test, color='green', lw=1.5, ls='--', label='Forecast periode testing')
    ax.plot(np.arange(n, n + 8), fc8v, color='orange', lw=1.8, ls='--', marker='o', ms=4,
            label='Forecast 8 minggu ke depan')
    ax.axvline(n_train - 0.5, color='gray', ls=':', lw=1.2)
    ax.set_title(f'{kab} (2021-2026)', fontsize=12)
    ax.set_xlabel('Minggu ke- (1 = minggu 1 2021)')
    ax.set_ylabel('Jumlah kasus')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

## 10. Interpretasi hasil

**Kenapa alpha kecil (mendekati 0)?**
Alpha dioptimasi otomatis dan hasilnya kecil untuk hampir semua kab/kota. Artinya data tidak memberi sinyal kuat untuk diikuti: kasus muncul sporadis di tengah dominasi nol. Model yang optimal adalah model yang "cuek", yaitu hampir selalu memprediksi nilai di sekitar rata-rata historis. Ini kesimpulan matematis, bukan kesalahan.

**Kenapa forecast-nya datar?**
SES tidak punya komponen tren maupun musiman, jadi ramalan untuk semua horizon ke depan sama dengan level terakhir. Untuk horizon pendek (8 minggu) ini masih wajar dipakai.

**Apa arti praktisnya?**
Model memprediksi kasus tetap rendah 8 minggu ke depan di semua kab/kota. Daerah yang historisnya nol (misal Kota Pagar Alam) diramal tetap nol. Ini sejalan dengan konteks eliminasi malaria: kasus sudah sangat jarang.

## 11. Sumber dan referensi

* Hyndman, R.J. & Athanasopoulos, G. Forecasting: Principles and Practice (bab Simple Exponential Smoothing). https://otexts.com/fpp3/
* Dokumentasi statsmodels SimpleExpSmoothing. https://www.statsmodels.org/
* Data: Sistem Kewaspadaan Dini dan Respon (SKDR), Kementerian Kesehatan RI (laporan analisa mingguan 2026) dan rekap dataset malaria Sumatera Selatan 2021-2025.
